# 13 · A confidence gate inside a workflow

## The problem

Real workflows have steps where an AI decision **does something**: approve a refund, send a reply, merge two records.
You have two bad defaults:
- **always automate**: fast, but every wrong call costs money or trust;
- **always ask a human**: safe, but slow, expensive, and your support team drowns in obvious cases.

What you want is: automate the **obvious yes**, route the **obvious no** to the safe path, and send only the
**genuinely unclear** middle to a person.

## Why Jev

That split only works if the model's number **means something**. Jev is a **System One** model: it returns typed
judgments with **calibrated probabilities** (a 0.7 is right about 70% of the time), in about 100 ms, for a fraction of
a cent. So code can draw lines on that probability and know what each band means. The policy itself goes into the
request as text, so changing the policy means editing a string, not retraining anything.

## What we'll build

```
refund request ──► Jev: P(eligible under POLICY) ──► gate(p) ──┬─► AUTO      p ≥ auto_at     approve, no human
     + policy          (one Noul, ~100 ms)            plain     ├─► CONFIRM   in between      a human clicks yes/no
                                                      Python    └─► FALLBACK  p < confirm_at  safe path (or call failed)
```

## What you'll learn
- How a `Noul` with explicit `NoulCriteria` pins down exactly what *yes* and *no* mean.
- The three-band gate pattern, and why a **failed call must never auto-approve**.
- How to **score once, then move thresholds for free**, and where to put the lines (in the gaps between scores).

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #5.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the API key (`TYPESAFE_API_KEY`) from the `.env` file at the repo root,
- creates the Jev client, `jev`, that every later cell uses,
- imports the Jev question types (here we only need `Noul` and `NoulCriteria`),
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so you can *see* the numbers.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

# Look for a .env file in this folder or any parent folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # the judge: fast, cheap, typed answers


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · The policy and the question

The **policy** is plain text that goes into the request's `state`. The question is a single `Noul`.

**How a `Noul` works:** `.noul` is P(yes), a number from 0 to 1. `NoulCriteria(true=..., false=...)` spells out exactly
what counts as *yes* and what counts as *no*. Here we deliberately put **"required facts are missing"** on the *no* side:
a vague request like *"refund pls"* should not drift towards approval just because nothing rules it out.

The `state` is a dict `{"policy": ..., "request": ...}`, and the backticks in the instruction (`` `policy` ``,
`` `request` ``) refer to those keys.

This cell only **defines** the policy and the question. Nothing is sent to Jev yet.

In [2]:
POLICY = ("Refunds are allowed within 30 days of delivery for unused items. "
          "Opened software, gift cards and final-sale items are not refundable. "
          "Items that arrived damaged are always refundable, at any time.")

GATE_QUESTION = {"eligible": Noul(
    instructions="Under `policy`, is `request` eligible for a refund?",
    criteria=NoulCriteria(true="The policy clearly allows this refund",
                          false="The policy clearly forbids it, or required facts are missing"),
)}
print("policy and question defined")

policy and question defined


## Step 2 · Wrap the call so failure is safe

`eligibility()` asks Jev and returns P(eligible). If the call fails for any reason (network, quota, bad key) it returns
`None` instead of raising. The gate treats `None` as **FALLBACK**, so an outage can slow the workflow down but can never
auto-approve a refund.

The guard at the top catches the most common workshop problem: a restarted kernel that forgot `GATE_QUESTION`.
We then try it once on an easy request.

In [3]:
def eligibility(request, client):
    """Returns P(eligible), or None if the call fails, so the workflow can fall back safely."""
    if "GATE_QUESTION" not in globals():
        raise RuntimeError("GATE_QUESTION is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")
    try:
        r = client.system_one(state={"policy": POLICY, "request": request}, questions=GATE_QUESTION)
        return r.nouls["eligible"].noul
    except Exception as e:
        print("  Jev call failed:", type(e).__name__)
        return None


p = eligibility("Bought headphones 10 days ago, never opened them, want my money back.", jev)
print(f"P(eligible) = {p:.2f}  {bar(p)}")

P(eligible) = 0.97  ███████████████████·


**What to notice**
- Unopened and well inside 30 days: the probability is **near 1**, an obvious yes.
- The answer is a single number, not a paragraph. Everything that happens next is our code's decision.

## Step 3 · The gate: three bands in plain code

`gate(p)` turns a probability into an action. Defaults:

| P(eligible) | band | what the workflow does |
|---|---|---|
| `≥ 0.90` (`auto_at`) | **AUTO** | refund immediately, no human |
| `0.30` to `0.90` | **CONFIRM** | a person sees it and clicks approve or deny |
| `< 0.30` (`confirm_at`), or the call failed | **FALLBACK** | the safe path: a polite "not eligible, here's why" or a standard support ticket |

> **Workshop note:** the first draft used `auto_at=0.85, confirm_at=0.40` and only clear-cut requests, so every
> request scored near 0 or near 1 and the CONFIRM band was always empty: moving the sliders did nothing. We added
> genuinely borderline requests in Step 4, and moved the defaults to **0.90 / 0.30** because those sit in the
> *gaps* between the clusters of scores we measured. A line drawn right on top of a cluster makes items flip bands from
> run to run. The policy and the question are unchanged.

In [4]:
AUTO_AT, CONFIRM_AT = 0.90, 0.30


def gate(p, auto_at=AUTO_AT, confirm_at=CONFIRM_AT):
    if p is None:
        return "FALLBACK"      # failed call -> safe path, never auto-approve
    if p >= auto_at:
        return "AUTO"
    if p >= confirm_at:
        return "CONFIRM"
    return "FALLBACK"


for test in (0.97, 0.6, 0.1, None):
    print(f"gate({test}) -> {gate(test)}")

gate(0.97) -> AUTO
gate(0.6) -> CONFIRM
gate(0.1) -> FALLBACK
gate(None) -> FALLBACK


## Step 4 · Score a batch of requests, once

Ten requests, from obvious to genuinely arguable:
- **clear yes**: unopened and recent; damaged on arrival (always refundable, even 3 months later);
- **borderline**: a vase taken out of the box but never used, on day 29; a sealed keyboard from a *clearance* section
  (is clearance the same as final sale?); a lampshade dent that *probably* happened in shipping; a blender used once;
- **clear no**: opened software, final sale, worn shoes, and a request with no facts at all.

We store the probabilities in `P` so the later steps can re-bucket them **without any new API calls**.

In [5]:
REQUESTS = [
    "Bought headphones 10 days ago, never opened them, want my money back.",
    "The blender arrived with a cracked jug. Delivered 3 months ago, I only just unboxed it.",
    "The vase was delivered 29 days ago. I took it out of the box to look at it but never used it.",
    "Bought a keyboard 2 weeks ago, still sealed, but it was in the clearance section.",
    "The lamp shade has a dent. I think it arrived like that, but I only noticed it this week. Delivered about 6 weeks ago.",
    "Delivered 12 days ago. I tried the blender once and it's too loud for me.",
    "I opened the photo-editing software but it's not what I expected.",
    "Jacket was on final sale, but it doesn't fit.",
    "Delivered 25 days ago, wore the shoes once around the house.",
    "refund pls",
]
P = {req: eligibility(req, jev) for req in REQUESTS}
for req, p in sorted(P.items(), key=lambda kv: -(kv[1] or 0)):
    print(f"  {bar(p or 0)} {(p or 0):.2f}  [{gate(p):^8}] {req[:72]}")

  ███████████████████· 0.97  [  AUTO  ] Bought headphones 10 days ago, never opened them, want my money back.
  ███████████████████· 0.96  [  AUTO  ] The blender arrived with a cracked jug. Delivered 3 months ago, I only j
  ██████████████······ 0.72  [CONFIRM ] The vase was delivered 29 days ago. I took it out of the box to look at 
  ██████████████······ 0.72  [CONFIRM ] Bought a keyboard 2 weeks ago, still sealed, but it was in the clearance
  ██████████████······ 0.72  [CONFIRM ] The lamp shade has a dent. I think it arrived like that, but I only noti
  ███████············· 0.33  [CONFIRM ] Delivered 12 days ago. I tried the blender once and it's too loud for me
  ███················· 0.16  [FALLBACK] Delivered 25 days ago, wore the shoes once around the house.
  ██·················· 0.11  [FALLBACK] refund pls
  █··················· 0.07  [FALLBACK] Jacket was on final sale, but it doesn't fit.
  █··················· 0.06  [FALLBACK] I opened the photo-editing software but it's no

**What to notice**
- All three bands are populated. The two obvious refunds are **AUTO**; opened software, final sale, worn shoes and
  *refund pls* are **FALLBACK**.
- The **CONFIRM** band holds exactly the cases a sensible support agent would want to look at: the vase on day 29, the
  clearance keyboard, the dent that might be shipping damage, the blender used once.
- The scores form **clusters** with gaps between them (near 1, roughly 0.65 to 0.85, a lone case around 0.35 to 0.45, and
  below 0.2). Good thresholds live in those gaps.
- *refund pls* lands low, not in the middle: that's the `NoulCriteria` at work ("required facts are missing" counts as
  *no*).

## Step 5 · Move the thresholds, no new API calls

Because we cached `P`, trying a different policy costs nothing. Three settings:

| setting | `auto_at` | `confirm_at` | intent |
|---|---|---|---|
| default | 0.90 | 0.30 | automate only the obvious yes; humans see everything arguable |
| fewer clicks | 0.90 | 0.55 | humans only see strong maybes; weak maybes take the safe path |
| aggressive | 0.60 | 0.30 | automate anything that's probably fine |

Each line lists the counts, then which requests **changed band** compared with the default.

In [6]:
SETTINGS = {"default": (0.90, 0.30), "fewer clicks": (0.90, 0.55), "aggressive": (0.60, 0.30)}
base = {req: gate(p) for req, p in P.items()}

for name, (auto_at, confirm_at) in SETTINGS.items():
    bands = {req: gate(p, auto_at, confirm_at) for req, p in P.items()}
    counts = [list(bands.values()).count(b) for b in ("AUTO", "CONFIRM", "FALLBACK")]
    print(f"{name:<13} auto>={auto_at:.2f} confirm>={confirm_at:.2f} ->  AUTO {counts[0]}  CONFIRM {counts[1]}  FALLBACK {counts[2]}")
    for req, b in bands.items():
        if b != base[req]:
            print(f"{'':<16}moved {base[req]:>8} -> {b:<8} {req[:58]}")

default       auto>=0.90 confirm>=0.30 ->  AUTO 2  CONFIRM 4  FALLBACK 4
fewer clicks  auto>=0.90 confirm>=0.55 ->  AUTO 2  CONFIRM 3  FALLBACK 5
                moved  CONFIRM -> FALLBACK Delivered 12 days ago. I tried the blender once and it's t
aggressive    auto>=0.60 confirm>=0.30 ->  AUTO 5  CONFIRM 1  FALLBACK 4
                moved  CONFIRM -> AUTO     The vase was delivered 29 days ago. I took it out of the b
                moved  CONFIRM -> AUTO     Bought a keyboard 2 weeks ago, still sealed, but it was in
                moved  CONFIRM -> AUTO     The lamp shade has a dent. I think it arrived like that, b


**What to notice**
- **Fewer clicks** raises the CONFIRM line: the blender-used-once request drops from CONFIRM to **FALLBACK**. Fewer
  human reviews, but a customer with an arguable case now gets the default "no".
- **Aggressive** lowers the AUTO line: the vase, the keyboard and the dented lampshade jump from CONFIRM to **AUTO**.
  Fewer clicks again, but you're now auto-refunding a clearance item that might be final sale.
- No setting is "correct". The right lines come from **labelled examples plus the cost of each mistake**: what does a
  wrong auto-refund cost you, versus a support agent's minute?

## Step 6 · Give it a face

Everything from here on is **presentation only**. This cell is a small style kit (colours, `box`, `badge`, `meter`) and
`form_app()`, which wires text boxes and a button to any function that returns HTML. Below it, a slider board re-buckets the cached `P` live, and the try-it form gates a new request.
**The decision logic above is untouched**: the UI only calls the same functions you've already seen run.

In [7]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

Drag the sliders: the cached probabilities are re-bucketed instantly, with no API calls. Try pulling `auto ≥` down
towards 0.6 and watch the middle cluster turn green, or pushing `confirm ≥` up past 0.5 and watch the lone
blender case turn red.

In [8]:
BAND_COLOR = {"AUTO": SAGE, "CONFIRM": AMBER, "FALLBACK": RED}


def gate_board(confirm_at=CONFIRM_AT, auto_at=AUTO_AT):
    rows = ""
    for req, p in sorted(P.items(), key=lambda kv: -(kv[1] or 0)):
        g = gate(p, auto_at, confirm_at)
        rows += (f'<div style="{CARD}padding:8px 12px;border-left:5px solid {BAND_COLOR[g]};">'
                 f'{badge(g, BAND_COLOR[g])} <b>{(p or 0):.2f}</b> &nbsp;{req}</div>')
    display(HTML(box(rows)))


widgets.interact(gate_board,
                 confirm_at=widgets.FloatSlider(value=CONFIRM_AT, min=0, max=1, step=0.05, description="confirm ≥"),
                 auto_at=widgets.FloatSlider(value=AUTO_AT, min=0, max=1, step=0.05, description="auto ≥"));

interactive(children=(FloatSlider(value=0.3, description='confirm ≥', max=1.0, step=0.05), FloatSlider(value=0…

## Step 7 · Try it yourself

Type a refund request and press **Gate it** (needs a live kernel). It uses the default lines (0.90 / 0.30). Good ones
to try in a workshop:
- `My kettle stopped working after a week, I've used it daily.` → a defect, not "arrived damaged", and it's been used:
  expect a middling score, usually **CONFIRM**.
- `Got the sweater about four weeks ago, tags still on, never worn.` → "about four weeks" is close to 30 days: a high
  score, near the AUTO line.
- `The TV screen was shattered when the box arrived yesterday.` → damaged on arrival: **AUTO**.
- `Can I return the $50 gift card my aunt gave me?` → explicitly excluded: **FALLBACK**.

In [9]:
def one_request_html(request):
    p = eligibility(request, jev)
    g = gate(p)
    return box(f'<div style="{CARD}border-left:5px solid {BAND_COLOR[g]};">{badge(g, BAND_COLOR[g])} '
               f'P(eligible) = <b>{(p or 0):.2f}</b><div style="margin-top:6px;">{request}</div></div>')


form_app([("request", "Refund request", "My kettle stopped working after a week, I've used it daily.", 2)],
         one_request_html, "Gate it")

Output()

## Recap and exercises

**Key ideas**
1. **Calibrated probability turns one model into three actions.** AUTO, CONFIRM and FALLBACK are just two lines drawn
   on P(yes).
2. **Failure is a band, too.** A failed call goes to the safe path and never auto-approves.
3. **Score once, tune for free.** Cache the probabilities and you can try any number of thresholds without new calls.
4. **Put the lines in the gaps.** A threshold that sits on a cluster of scores makes decisions flip between runs. Set
   them from labelled data and the cost of each error.

**Exercises**
- Change the policy to *"within 14 days"* (edit the string, nothing else) and re-run Steps 4 and 5. Which requests move?
- Add a sentence to `POLICY`: *"Clearance items count as final sale."* Where does the keyboard go now?
- Label each request in `REQUESTS` with what *you* would decide, then write a loop that counts wrong AUTOs and wrong
  FALLBACKs for each setting in `SETTINGS`.
- Give the CONFIRM band a cost (say 1 minute of agent time) and a wrong AUTO a cost (say $40), and pick the setting with
  the lowest total.